# multilingual-e5-large — ClusterLLM (LLM-guided dendrogram cut)

Flagged in the literature as significantly more complex and compute-intensive than contrastive
methods like SCCL (`clustering_e5_SCCL.ipynb`) — and structurally very different: ClusterLLM does
**not** fine-tune anything or use the LLM as an embedder. It:

1. Runs ordinary hierarchical (agglomerative) clustering first, producing a full dendrogram —
   exactly the same Ward-linkage clustering used in `e5/clustering_e5_Agglomerative.ipynb`.
2. Uses an LLM to answer **triplet questions** ("is article A more similar to article B or
   article C?") at candidate merge points in that dendrogram, and uses its answers to decide
   *where to cut* the hierarchy into a flat set of clusters — i.e., the LLM judges cluster
   *granularity*, not similarity at the embedding level.

**Deviation from the ClusterLLM paper, and why**: the original method queries a large proprietary
LLM (e.g. GPT-family) via API, with entropy-based sampling of *which* triplets are most
informative, across the *entire* dendrogram. That needs either a paid API budget this project
doesn\'t have configured, or an enormous number of local LLM calls. This notebook instead:

- Runs entirely on a small **open, locally-loaded instruction-tuned multilingual LLM**
  (`Qwen/Qwen2.5-3B-Instruct` — multilingual across 100+ languages including Sinhala, small enough
  for a single Kaggle GPU) so nothing needs an external API key or budget.
- Walks a **coarse, fixed set of candidate cluster counts** (the same style of candidate list used
  to sweep k in every other notebook here) from many clusters down to few, rather than an
  entropy-guided search over every possible triplet.
- At each step, samples a **small, bounded number of merge events** (≤3) to query the LLM about,
  instead of exhaustively checking every merge — keeping total LLM calls in the tens, not
  thousands.
- Uses a simple **greedy stopping rule**: keep accepting coarser cuts (fewer clusters) as long as
  the LLM agrees the just-merged sub-clusters are more similar to each other than to an unrelated
  cluster; stop and cut at the last point where it still agreed once agreement drops.

This keeps the core idea faithful — **LLM as judge of where to cut, not as the embedder** — while
keeping compute and LLM-call count bounded for a project-scale comparison rather than a
publication-scale one.

In [1]:
!pip install -q umap-learn accelerate

In [2]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
import random
from pathlib import Path

SEED = 42
np.random.seed(SEED)
rng_py = random.Random(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_clusterllm")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/e5_clusterllm


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

df shape after text cleaning: (1999, 7)


,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [9]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

Documents: 1999


'passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහනුවර දිස්ත්\u200dරික් මන්ත්\u200dරී ජගත් මනුවර්ණ මහතා:-(ජා.ජ.බ) පාර්ලිමේන්තුවේදී පැවසීය.\nදූෂණයට විරුද්ධ වීම භයානක බවත් දූෂණයට විරුද්ධ නොවී සිටීම ඊට වඩා භයානක බවත් අනුර දිසානායක ජනාධිපති\xa0 එක්සත් ජාතීන්ගේ මහා මණ්ඩලයේ අමතමින් ප්\u200dරකාශ කළා.එය අප නැවත අවධාරණය කළ යුතුයි.අපි දේශපාලන පලි ගැනීම් කරනවා යැයි චෝදනා කරනවා.නමුත් ඇත්ත ඒක නෙමෙයි.මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය එක\xa0 ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම ඊට පස'

In [10]:
# Embed with multilingual-e5-large (frozen — no fine-tuning here, unlike clustering_e5_SCCL.ipynb),
# mean pooling — established as the best strategy for this model on this corpus
import torch
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 228.69 MiB is free. Process 105 has 2.79 GiB memory in use. Process 174 has 2.79 GiB memory in use. Process 229 has 2.79 GiB memory in use. Process 281 has 2.79 GiB memory in use. Process 342 has 2.79 GiB memory in use. Process 397 has 102.00 MiB memory in use. Process 439 has 102.00 MiB memory in use. Process 472 has 102.00 MiB memory in use. Including non-PyTorch memory, this process has 102.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Embedding separation score (comparable across notebooks)
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sims = cosine_similarity(embeddings[sample_idx])
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()
print(f"Separation score: {separation_score:.4f}")

In [ ]:
# Reduce dimensionality with UMAP (same settings used across this project\'s UMAP-based
# notebooks) and build the full agglomerative (Ward) dendrogram
from umap import UMAP
from scipy.cluster.hierarchy import linkage, fcluster

umap_model = UMAP(n_neighbors=3, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
reduced_embeddings = umap_model.fit_transform(embeddings)
print(reduced_embeddings.shape)

Z = linkage(reduced_embeddings, method="ward")
print("Dendrogram built:", Z.shape[0], "merge steps for", len(reduced_embeddings), "articles")


def clusters_at_k(k):
    return fcluster(Z, t=k, criterion="maxclust")

In [ ]:
# Load a small, locally-run, multilingual instruction-tuned LLM to act as the triplet judge.
# Qwen2.5-3B-Instruct: multilingual across 100+ languages (including Sinhala), small enough for a
# single Kaggle GPU, and needs no API key/budget.
from transformers import AutoModelForCausalLM, pipeline

llm_model_name = "Qwen/Qwen2.5-3B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name, torch_dtype=torch.float16, device_map="auto",
)
llm = pipeline("text-generation", model=llm_model, tokenizer=llm_tokenizer)


def ask_triplet(title_a, title_b, title_c, max_new_tokens=5):
    """Ask the LLM: is article A more similar to article B or article C? Returns "B" or "C"
    (falls back to a coin flip if the response can\'t be parsed, so one bad generation can\'t
    silently bias the whole run)."""
    messages = [
        {"role": "system", "content": "You are a news-topic similarity judge. Answer with exactly one letter: B or C. No explanation."},
        {"role": "user", "content": (
            f"Article A: \"{title_a}\"\n"
            f"Article B: \"{title_b}\"\n"
            f"Article C: \"{title_c}\"\n"
            "Which article, B or C, is about a more similar news topic/event to Article A?"
        )},
    ]
    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = llm(prompt, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"]
    reply = output[len(prompt):].strip().upper()
    if "B" in reply and "C" not in reply:
        return "B"
    if "C" in reply and "B" not in reply:
        return "C"
    if "B" in reply and "C" in reply:
        return "B" if reply.index("B") < reply.index("C") else "C"
    return rng_py.choice(["B", "C"])  # unparseable — don\'t let it silently favor one side


# Smoke check
print(ask_triplet("Election results announced", "President wins by narrow margin", "New species of frog discovered"))

In [ ]:
# Find merge events between two candidate cluster counts: which finer-grained clusters
# (at k_prev) got combined into one coarser cluster (at k_next)?
def find_merge_events(labels_prev, labels_next, max_events=3, rng=None):
    df_map = pd.DataFrame({"prev": labels_prev, "next": labels_next})
    events = []
    for next_id, group in df_map.groupby("next"):
        prev_ids = list(group["prev"].unique())
        if len(prev_ids) > 1:
            events.append((next_id, prev_ids))
    if rng is not None:
        rng.shuffle(events)
    return events[:max_events]

In [ ]:
# Walk the dendrogram from many clusters down to few, asking the LLM at each step whether
# the merges being proposed actually make sense, and cut where it stops agreeing.
candidate_k = sorted(set(list(range(10, 100, 15)) + list(range(100, 700, 100))), reverse=True)
candidate_k = [k for k in candidate_k if k < len(reduced_embeddings)]

AGREEMENT_THRESHOLD = 0.5
MAX_EVENTS_PER_STEP = 3

titles = df["title"].tolist()
prev_labels = clusters_at_k(candidate_k[0])
chosen_k = candidate_k[0]
agreement_history = []

for k in candidate_k[1:]:
    next_labels = clusters_at_k(k)
    events = find_merge_events(prev_labels, next_labels, max_events=MAX_EVENTS_PER_STEP, rng=rng_py)

    agreements = []
    for next_id, prev_ids in events:
        prev_a, prev_b = prev_ids[0], prev_ids[1]
        idx_a = int(np.where(prev_labels == prev_a)[0][0])
        idx_b = int(np.where(prev_labels == prev_b)[0][0])
        other_next_ids = [nid for nid in np.unique(next_labels) if nid != next_id]
        distractor_next = rng_py.choice(list(other_next_ids))
        idx_c = int(np.where(next_labels == distractor_next)[0][0])

        answer = ask_triplet(titles[idx_a], titles[idx_b], titles[idx_c])
        agreements.append(answer == "B")

    agreement_rate = float(np.mean(agreements)) if agreements else 1.0
    agreement_history.append({"k": k, "agreement_rate": agreement_rate, "n_events": len(events)})
    print(f"k={k:>4}  events={len(events)}  agreement_rate={agreement_rate:.2f}")

    if agreement_rate < AGREEMENT_THRESHOLD:
        print(f"Agreement dropped below {AGREEMENT_THRESHOLD} at k={k}; cutting at previous k={chosen_k}")
        break

    chosen_k = k
    prev_labels = next_labels

print(f"\nChosen k (LLM-guided cut): {chosen_k}")
agreement_df = pd.DataFrame(agreement_history)
agreement_df

In [ ]:
# Final clustering at the LLM-chosen cut
final_labels = clusters_at_k(chosen_k)
df["cluster_id"] = final_labels

print(pd.Series(final_labels).value_counts().head(20))
print("Number of clusters found:", len(set(final_labels)))

In [ ]:
# Clustering evaluation
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

n_clusters = len(set(final_labels))
sil = silhouette_score(reduced_embeddings, final_labels, metric="cosine")
dbi = davies_bouldin_score(reduced_embeddings, final_labels)
ch = calinski_harabasz_score(reduced_embeddings, final_labels)

print(f"Model: {embedding_model_name} + UMAP + Agglomerative(Ward) + ClusterLLM cut selection")
print(f"Articles: {len(df)} | Clusters: {n_clusters} | Noise ratio: 0.00% (n/a for this algorithm)")
print(f"Silhouette Score (cosine): {sil:.4f}")
print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")

scores_path = RESULTS_DIR / "e5_clusterllm_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "UMAP+Agglomerative(Ward)+ClusterLLM",
    "judge_llm": llm_model_name,
    "embedding_dim": embeddings.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": 0.0,
    "silhouette": round(sil, 4),
    "davies_bouldin": round(dbi, 4),
    "calinski_harabasz": round(ch, 2),
    "chosen_k": chosen_k,
    "agreement_threshold": AGREEMENT_THRESHOLD,
}
pd.DataFrame([row]).to_csv(scores_path, index=False)

agreement_path = RESULTS_DIR / "e5_clusterllm_agreement_history.csv"
agreement_df.to_csv(agreement_path, index=False)
print(f"Saved scores to {scores_path}")
print(f"Saved LLM agreement history to {agreement_path}")

In [ ]:
# Inspect sample titles per cluster (first few)
for cluster_id, group in list(df.groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_clusterllm_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")